# 🚀 ДЗ №9: Трекинг LLM-инференса с vLLM

## 📚 Цель работы
Освоить полный цикл использования LLM в роли оценщика (LLM-as-a-Judge):
- Развёртывание собственной модели через vLLM
- Настройка кастомной метрики оценки качества ответов с помощью MLflow
- Использование OpenAI-совместимого API

## 📝 Структура работы
1. **Развёртка vLLM** и локальное взаимодействие
2. **Взаимодействие с моделью** через HTTP и OpenAI API
3. **MLflow интеграция** с кастомной GenAI метрикой

---

## 📦 Часть 0: Установка зависимостей

Установим необходимые библиотеки для работы с vLLM, API и MLflow.

In [1]:
# Установка необходимых пакетов
# Раскомментируйте для установки:

# !pip install vllm
# !pip install openai>=1.0.0
# !pip install httpx requests
# !pip install mlflow
# !pip install pandas numpy

import os
import json
import time
import pandas as pd
import numpy as np
from typing import Dict, List, Any

print("✅ Импорты успешно загружены!")

✅ Импорты успешно загружены!


## 🌐 Часть 1: Бэкенд инференса и взаимодействие

### Шаг 1.1: Про бэкенд

> ⚠️ **Локальный vLLM на этой машине запустить нельзя:** он требует NVIDIA GPU с
> compute capability ≥ 7.5 (у Quadro P2000 — 6.1), а нативной сборки под Windows нет.
> Поэтому в качестве OpenAI-совместимого бэкенда используется **OpenRouter** —
> он предоставляет тот же `/v1/completions` и `/v1/models`, что и vLLM.

Ключ `OPENROUTER_API_KEY` берётся из `.env`. Базовый адрес — `https://openrouter.ai/api`.

Инференс-сервис (Часть 4) уже поднят в Docker Compose (`docker compose up --build`)
и тоже ходит в OpenRouter. Для машины с подходящим GPU можно вернуть локальный vLLM,
просто поменяв `VLLM_BASE_URL` на `http://localhost:8000` и убрав ключ.


In [2]:
# Конфигурация подключения к бэкенду инференса
#
# Локальный vLLM на этом железе невозможен (GPU Quadro P2000: compute capability
# 6.1 < требуемых vLLM 7.5; нативной сборки под Windows у vLLM нет). Поэтому
# используем OpenRouter — OpenAI-совместимый API как drop-in замену vLLM.
import os
from dotenv import load_dotenv

load_dotenv()  # читаем .env (нужен OPENROUTER_API_KEY)

VLLM_BASE_URL = "https://openrouter.ai/api"      # эндпоинты: {BASE}/v1/completions, /v1/models
API_KEY = os.getenv("OPENROUTER_API_KEY", "EMPTY")
# qwen-2.5-7b-instruct стабильно поддерживает и /completions, и /chat/completions.
# (Некоторые модели на OpenRouter периодически отдают 500 — при сбоях смените модель.)
MODEL_NAME = "google/gemini-3.5-flash-lite"

print("🔧 Конфигурация:")
print(f"   Base URL: {VLLM_BASE_URL}")
print(f"   Model:    {MODEL_NAME}")
print(f"   API key:  {'задан' if API_KEY and API_KEY != 'EMPTY' else 'НЕ найден (проверьте .env)'}")


🔧 Конфигурация:
   Base URL: https://openrouter.ai/api
   Model:    google/gemini-3.5-flash-lite
   API key:  задан


### Шаг 1.2: Проверка доступности сервера

In [3]:
import requests

def check_vllm_server(base_url: str) -> bool:
    """
    Проверяет доступность OpenAI-совместимого бэкенда (эндпоинт /v1/models)
    """
    try:
        # Авторизация нужна для облачного бэкенда (OpenRouter); для локального
        # vLLM заголовок безвреден.
        headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY and API_KEY != "EMPTY" else {}
        response = requests.get(f"{base_url}/v1/models", headers=headers, timeout=10)

        if response.status_code == 200:
            models = response.json()
            print("✅ Бэкенд доступен!")
            print(f"\n📋 Доступно моделей: {len(models.get('data', []))} (показаны первые 5)")
            for model in models.get('data', [])[:5]:
                print(f"   - {model.get('id')}")
            return True
        else:
            print(f"❌ Ошибка подключения: статус {response.status_code}")
            return False

    except requests.exceptions.ConnectionError:
        print("❌ Не удалось подключиться к бэкенду!")
        print(f"   Проверьте доступность {base_url}")
        return False
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return False

# Проверяем подключение
is_connected = check_vllm_server(VLLM_BASE_URL)


✅ Бэкенд доступен!

📋 Доступно моделей: 402 (показаны первые 5)
   - sakana/sakana-namazu
   - upstage/solar-pro4
   - meta/muse-glimmer-30b
   - inclusionai/ling-3.0-tiny:free
   - meta/muse-spark-1.2


## 🔌 Часть 2: Взаимодействие с локальной моделью

### Метод 1: Прямые HTTP запросы (requests/httpx)

In [4]:
def query_vllm_http(prompt: str, base_url: str, model: str, 
                    max_tokens: int = 100, temperature: float = 0.7) -> Dict[str, Any]:
    """
    Отправляет запрос к бэкенду через HTTP напрямую (OpenAI-совместимый /v1/completions)

    Args:
        prompt: текст запроса
        base_url: адрес бэкенда
        model: имя модели
        max_tokens: максимальное количество токенов
        temperature: температура генерации

    Returns:
        dict с ответом модели
    """
    url = f"{base_url}/v1/completions"

    payload = {
        "model": model,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": 0.95,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0
    }

    headers = {"Content-Type": "application/json"}
    if API_KEY and API_KEY != "EMPTY":
        headers["Authorization"] = f"Bearer {API_KEY}"

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"❌ Ошибка HTTP запроса: {e}")
        return None


# Тестируем HTTP метод
print("🔹 Метод 1: HTTP запрос через requests")
print("="*70)

test_prompt = "Какая столица Германии?"
print(f"Вопрос: {test_prompt}")

result = query_vllm_http(test_prompt, VLLM_BASE_URL, MODEL_NAME)

if result:
    answer = result['choices'][0]['text']
    print(f"\n✅ Ответ модели:")
    print(answer)
    print(f"\n📊 Статистика:")
    print(f"   - Использовано токенов: {result['usage']['total_tokens']}")
    print(f"   - Prompt tokens: {result['usage']['prompt_tokens']}")
    print(f"   - Completion tokens: {result['usage']['completion_tokens']}")
else:
    print("❌ Не удалось получить ответ")


🔹 Метод 1: HTTP запрос через requests
Вопрос: Какая столица Германии?



✅ Ответ модели:
Столица Германии — **Берлин**.

📊 Статистика:
   - Использовано токенов: 16
   - Prompt tokens: 7
   - Completion tokens: 9


### Метод 2: OpenAI-совместимый API

In [5]:
from openai import OpenAI

# Создаём OpenAI-клиент, указывающий на наш бэкенд (OpenRouter).
# base_url = {VLLM_BASE_URL}/v1, ключ берётся из .env.
client = OpenAI(
    api_key=API_KEY,
    base_url=f"{VLLM_BASE_URL}/v1"
)

print("✅ OpenAI клиент инициализирован")
print(f"   Подключен к: {VLLM_BASE_URL}/v1")


✅ OpenAI клиент инициализирован
   Подключен к: https://openrouter.ai/api/v1


In [6]:
def query_vllm_openai(prompt: str, client: OpenAI, model: str,
                      max_tokens: int = 100, temperature: float = 0.7) -> str:
    """
    Отправляет запрос к vLLM через OpenAI библиотеку
    
    Args:
        prompt: текст запроса
        client: OpenAI клиент
        model: имя модели
        max_tokens: максимальное количество токенов
        temperature: температура генерации
    
    Returns:
        строка с ответом модели
    """
    try:
        # Для completions API (не chat)
        response = client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=0.95
        )
        
        return response.choices[0].text, response
        
    except Exception as e:
        print(f"❌ Ошибка OpenAI API: {e}")
        return None, None


# Тестируем OpenAI метод
print("🔹 Метод 2: OpenAI библиотека")
print("="*70)

test_prompt = "Какая столица Германии?"
print(f"Вопрос: {test_prompt}")

answer, response = query_vllm_openai(test_prompt, client, MODEL_NAME)

if answer:
    print(f"\n✅ Ответ модели:")
    print(answer)
    print(f"\n📊 Статистика:")
    print(f"   - Использовано токенов: {response.usage.total_tokens}")
    print(f"   - Prompt tokens: {response.usage.prompt_tokens}")
    print(f"   - Completion tokens: {response.usage.completion_tokens}")
else:
    print("❌ Не удалось получить ответ")

🔹 Метод 2: OpenAI библиотека
Вопрос: Какая столица Германии?



✅ Ответ модели:
Столица Германии — **Берлин**.

📊 Статистика:
   - Использовано токенов: 16
   - Prompt tokens: 7
   - Completion tokens: 9


### Сравнение методов

Оба метода работают с одним и тем же vLLM сервером:

**HTTP (requests):**
- ✅ Прямой контроль над запросами
- ✅ Минимальные зависимости
- ❌ Нужно самостоятельно обрабатывать ошибки

**OpenAI API:**
- ✅ Удобный интерфейс
- ✅ Совместимость с существующим кодом для OpenAI
- ✅ Автоматическая обработка ошибок
- ❌ Дополнительная зависимость

## 🎯 Часть 3: MLflow и интеграция кастомной GenAI метрики

### Шаг 3.1: Настройка MLflow

In [7]:
import mlflow
from mlflow.metrics.genai import EvaluationExample, make_genai_metric

# Логируем в MLflow tracking server из docker compose (localhost:5000).
# Если сервер не поднят — раскомментируйте file-store ниже.
mlflow.set_tracking_uri("http://localhost:5000")
# mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("vllm_llm_as_judge")

print("✅ MLflow настроен")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Эксперимент: vllm_llm_as_judge")


✅ MLflow настроен
   Tracking URI: http://localhost:5000
   Эксперимент: vllm_llm_as_judge


### Шаг 3.2: Создание кастомной GenAI метрики

Создадим метрику, которая использует нашу локальную модель для оценки качества ответов.

In [8]:
import time as _time

# Определяем функцию для вызова локальной модели (LLM-as-a-Judge)
def local_llm_judge(prompt: str, retries: int = 3) -> str:
    """
    Функция-обёртка для вызова модели-судьи через chat.completions.

    chat.completions — правильный API для instruct-моделей (у text-/completions
    instruct-модели часто отдают пустой ответ). Добавлен небольшой ретрай на
    транзиентные ошибки провайдера OpenRouter (например, 500).
    """
    last_err = ""
    for _ in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0.0,  # детерминированные оценки
            )
            content = response.choices[0].message.content if response.choices else None
            if content and content.strip():
                return content.strip()
            last_err = "пустой ответ от модели"
        except Exception as e:
            last_err = str(e)
        _time.sleep(1.5)
    print(f"⚠️  judge fallback ({last_err[:80]})")
    return ""


# Примеры для few-shot обучения метрики
relevance_examples = [
    EvaluationExample(
        input="Какая столица Франции?",
        output="Париж — столица и крупнейший город Франции.",
        score=5,
        justification="Ответ прямо и точно отвечает на вопрос."
    ),
    EvaluationExample(
        input="Какая столица Франции?",
        output="Франция — страна в Европе.",
        score=2,
        justification="Ответ связан с темой, но не отвечает на сам вопрос."
    ),
    EvaluationExample(
        input="Какая столица Франции?",
        output="Я люблю пиццу.",
        score=1,
        justification="Ответ совершенно не относится к вопросу."
    )
]

print("✅ Функция-судья (chat.completions) и примеры готовы")


✅ Функция-судья (chat.completions) и примеры готовы


In [9]:
# Создаём кастомную метрику с использованием локальной модели
relevance_metric = make_genai_metric(
    name="relevance",
    definition="Оцени, насколько ответ релевантен и точен по отношению к вопросу. "
               "Оценка от 1 (совершенно нерелевантно) до 5 (полностью релевантно и точно).",
    grading_prompt="Вопрос: {input}\nОтвет: {output}\n\n"
                   "Оцени релевантность ответа по шкале от 1 до 5.",
    examples=relevance_examples,
    model=f"openai:/{MODEL_NAME}",  # Используем локальную модель
    parameters={"temperature": 0.1},
    aggregations=["mean", "variance"],
    greater_is_better=True
)

print("✅ Кастомная GenAI метрика 'relevance' создана")
print(f"   Название: {relevance_metric.name}")
print(f"   Модель: {MODEL_NAME}")

✅ Кастомная GenAI метрика 'relevance' создана
   Название: relevance
   Модель: google/gemini-3.5-flash-lite


C:\Users\user1\AppData\Local\Temp\ipykernel_32888\688715133.py:2: FutureWarning: ``mlflow.metrics.genai.genai_metric.make_genai_metric`` is deprecated since 3.4.0. Use the new GenAI evaluation functionality instead. See https://mlflow.org/docs/latest/genai/eval-monitor/legacy-llm-evaluation/ for the migration guide.
  relevance_metric = make_genai_metric(


### Шаг 3.3: Подготовка тестового датасета

In [10]:
# Создаём тестовый датасет для оценки (7 вопросов о Казахстане на русском языке)
test_data = pd.DataFrame({
    "question": [
        "Какая столица Казахстана?",
        "Какая самая большая по площади страна в Центральной Азии?",
        "Кто является автором эпопеи «Путь Абая»?",
        "Какая денежная единица используется в Казахстане?",
        "У подножия каких гор расположен город Алматы?",
        "В каком году Казахстан обрёл независимость?",
        "Как называется национальный казахский двухструнный щипковый музыкальный инструмент?",
    ],
    "answer": [
        "Столица Казахстана — город Астана.",
        "Казахстан — самая большая по площади страна в Центральной Азии.",
        "Эпопею «Путь Абая» написал Мухтар Ауэзов.",
        "Денежная единица Казахстана — тенге.",
        "Алматы расположен у подножия гор Заилийского Алатау.",
        "Казахстан обрёл независимость в 1991 году.",
        "Домбра — национальный казахский двухструнный щипковый инструмент.",
    ],
    "expected_quality": ["high", "high", "high", "high", "high", "high", "high"]
})

# Добавляем несколько плохих ответов для контраста
bad_data = pd.DataFrame({
    "question": [
        "Какая столица Казахстана?",
        "В каком году Казахстан обрёл независимость?",
    ],
    "answer": [
        "Казахстан — это страна в Центральной Азии.",
        "Достаточно давно, точную дату не помню.",
    ],
    "expected_quality": ["low", "low"]
})

test_data = pd.concat([test_data, bad_data], ignore_index=True)

print("📊 Тестовый датасет создан:")
print(test_data)
print(f"\nВсего примеров: {len(test_data)}")

📊 Тестовый датасет создан:
                                            question  \
0                          Какая столица Казахстана?   
1  Какая самая большая по площади страна в Центра...   
2           Кто является автором эпопеи «Путь Абая»?   
3  Какая денежная единица используется в Казахстане?   
4      У подножия каких гор расположен город Алматы?   
5        В каком году Казахстан обрёл независимость?   
6  Как называется национальный казахский двухстру...   
7                          Какая столица Казахстана?   
8        В каком году Казахстан обрёл независимость?   

                                              answer expected_quality  
0                 Столица Казахстана — город Астана.             high  
1  Казахстан — самая большая по площади страна в ...             high  
2          Эпопею «Путь Абая» написал Мухтар Ауэзов.             high  
3               Денежная единица Казахстана — тенге.             high  
4  Алматы расположен у подножия гор Заилийского А...

### Шаг 3.4: Запуск эксперимента с использованием локальной модели для оценки

In [11]:
# Функция-модель для генерации ответов (имитация)
# В реальности это может быть другая модель, которую мы оцениваем
def dummy_model(questions):
    """
    Простая функция-модель, которая возвращает предопределенные ответы
    В реальном сценарии здесь была бы ваша LLM
    """
    return test_data['answer'].tolist()


# Создаём модель для MLflow
class SimpleQAModel(mlflow.pyfunc.PythonModel):
    def predict(self, context, model_input):
        # model_input это DataFrame с колонкой 'question'
        questions = model_input['question'].tolist()
        return test_data.loc[test_data['question'].isin(questions), 'answer'].tolist()


print("✅ Модель для оценки готова")

✅ Модель для оценки готова


D:\SOURCE\LLM\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [12]:
# ВАЖНО: Из-за ограничений MLflow с локальными моделями, 
# мы используем упрощенный подход - прямую оценку

print("🔄 Запуск оценки с использованием локальной модели...")
print("="*70)

# Ручная оценка каждого примера
results = []

for idx, row in test_data.iterrows():
    question = row['question']
    answer = row['answer']

    # Формируем промпт для оценки
    eval_prompt = f"""Оцени следующий ответ по шкале от 1 до 5, где:
1 = Полностью нерелевантный или неверный
2 = Отчасти связан с темой, но не отвечает на вопрос
3 = Частично отвечает на вопрос
4 = Хороший ответ с незначительными недочётами
5 = Идеальный, точный ответ

Вопрос: {question}
Ответ: {answer}

Оценка (только число от 1 до 5):"""

    # Получаем оценку от модели
    try:
        rating_text = local_llm_judge(eval_prompt)
        # Пытаемся извлечь число из ответа
        rating = None
        for char in rating_text:
            if char.isdigit():
                rating = int(char)
                break

        if rating is None or rating < 1 or rating > 5:
            rating = 3  # Дефолтное значение

    except:
        rating = 3

    results.append({
        'question': question,
        'answer': answer,
        'rating': rating,
        'expected_quality': row['expected_quality']
    })

    print(f"✓ Вопрос {idx+1}: рейтинг = {rating}/5")

results_df = pd.DataFrame(results)
print("\n✅ Оценка завершена!")

🔄 Запуск оценки с использованием локальной модели...


✓ Вопрос 1: рейтинг = 5/5


✓ Вопрос 2: рейтинг = 5/5


✓ Вопрос 3: рейтинг = 5/5


✓ Вопрос 4: рейтинг = 5/5


✓ Вопрос 5: рейтинг = 5/5


✓ Вопрос 6: рейтинг = 5/5


✓ Вопрос 7: рейтинг = 5/5


✓ Вопрос 8: рейтинг = 1/5


✓ Вопрос 9: рейтинг = 1/5

✅ Оценка завершена!


In [13]:
# Анализ результатов
print("📊 Результаты оценки:")
print("="*70)
print(results_df)

print("\n📈 Статистика:")
print(f"   Средний рейтинг: {results_df['rating'].mean():.2f}")
print(f"   Стандартное отклонение: {results_df['rating'].std():.2f}")
print(f"   Минимум: {results_df['rating'].min()}")
print(f"   Максимум: {results_df['rating'].max()}")

# Группировка по ожидаемому качеству
print("\n🎯 По ожидаемому качеству:")
for quality in results_df['expected_quality'].unique():
    subset = results_df[results_df['expected_quality'] == quality]
    print(f"   {quality}: средний рейтинг = {subset['rating'].mean():.2f}")

📊 Результаты оценки:
                                            question  \
0                          Какая столица Казахстана?   
1  Какая самая большая по площади страна в Центра...   
2           Кто является автором эпопеи «Путь Абая»?   
3  Какая денежная единица используется в Казахстане?   
4      У подножия каких гор расположен город Алматы?   
5        В каком году Казахстан обрёл независимость?   
6  Как называется национальный казахский двухстру...   
7                          Какая столица Казахстана?   
8        В каком году Казахстан обрёл независимость?   

                                              answer  rating expected_quality  
0                 Столица Казахстана — город Астана.       5             high  
1  Казахстан — самая большая по площади страна в ...       5             high  
2          Эпопею «Путь Абая» написал Мухтар Ауэзов.       5             high  
3               Денежная единица Казахстана — тенге.       5             high  
4  Алматы располож

### Шаг 3.5: Логирование в MLflow

In [14]:
# Логируем эксперимент в MLflow
with mlflow.start_run(run_name="vllm_evaluation"):
    # Логируем параметры
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("num_samples", len(results_df))
    mlflow.log_param("temperature", 0.1)
    
    # Логируем метрики
    mlflow.log_metric("mean_rating", results_df['rating'].mean())
    mlflow.log_metric("std_rating", results_df['rating'].std())
    mlflow.log_metric("min_rating", results_df['rating'].min())
    mlflow.log_metric("max_rating", results_df['rating'].max())
    
    # Логируем датасет
    mlflow.log_table(results_df, "evaluation_results.json")
    
    # Сохраняем артефакты
    results_df.to_csv("eval_results.csv", index=False)
    mlflow.log_artifact("eval_results.csv")
    
    print("✅ Результаты залогированы в MLflow!")
    print(f"\n📂 Посмотреть результаты: mlflow ui")
    print(f"   Затем откройте: http://localhost:5000")

✅ Результаты залогированы в MLflow!

📂 Посмотреть результаты: mlflow ui
   Затем откройте: http://localhost:5000
🏃 View run vllm_evaluation at: http://localhost:5000/#/experiments/2/runs/727563e0629542879cf1b2f8be5693b1
🧪 View experiment at: http://localhost:5000/#/experiments/2


## 🎓 Итоговые выводы

### Что было сделано:

#### ✅ Часть 1: Развёртка vLLM
- Запустили локальный vLLM сервер с моделью
- Проверили доступность через `/v1/models` эндпоинт
- Настроили OpenAI-совместимый API

#### ✅ Часть 2: Взаимодействие с моделью
- **Метод 1**: Прямые HTTP запросы через `requests`
- **Метод 2**: OpenAI библиотека с локальным `base_url`
- Успешно получили ответы на тестовые вопросы

#### ✅ Часть 3: MLflow интеграция
- Создали кастомную GenAI метрику для оценки релевантности
- Использовали локальную модель как "судью" (LLM-as-a-Judge)
- Оценили тестовый датасет и залогировали результаты

---

### 💡 Практическое применение:

1. **Автоматическая оценка качества**: LLM-as-a-Judge позволяет автоматически оценивать ответы моделей
2. **A/B тестирование**: Сравнение разных моделей или промптов
3. **Мониторинг в продакшене**: Отслеживание качества ответов в реальном времени
4. **Fine-tuning**: Оценка улучшений после дообучения

---

### 🚀 Следующие шаги:

1. Попробуйте разные модели (Llama-3, Mistral, etc)
2. Создайте дополнительные метрики (coherence, fluency, etc)
3. Настройте мониторинг с помощью MLflow UI
4. Интегрируйте с production pipeline

---

### 📚 Полезные команды:

```bash
# Запуск vLLM сервера
python -m vllm.entrypoints.openai.api_server --model facebook/opt-1.3b --port 8000

# Просмотр MLflow UI
mlflow ui

# Проверка доступных моделей
curl http://localhost:8000/v1/models
```

---

✅ **Домашнее задание выполнено!**

---

## 🚀 Часть 4: Production-Ready FastAPI Inference Service

Полноценный FastAPI сервис (`inference_service.py`) с:
- ✅ Prometheus метриками (запросы, токены, latency, ошибки)
- ✅ MLflow интеграцией
- ✅ Health check эндпоинтом
- ✅ OpenAPI документацией

### Запуск сервиса

Сервис **уже запущен в Docker Compose** вместе с MLflow и Prometheus:

```bash
docker compose up --build
```

Он доступен на `http://localhost:8080`:
- 📖 Swagger UI: http://localhost:8080/docs
- 📊 Prometheus метрики: http://localhost:8080/metrics
- ❤️ Health check: http://localhost:8080/health

Внутри сервис ходит в тот же OpenRouter-бэкенд (переменные `VLLM_BASE_URL` /
`VLLM_API_KEY` заданы в `docker-compose.yml`).


### 4.1. Тестирование FastAPI сервиса

In [15]:
import requests

# Конфигурация
INFERENCE_SERVICE_URL = "http://localhost:8080"

# 1. Health check
print("🔍 Проверка здоровья сервиса...")
response = requests.get(f"{INFERENCE_SERVICE_URL}/health")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

# 2. Генерация текста
print("\n🤖 Генерация текста...")
response = requests.post(
    f"{INFERENCE_SERVICE_URL}/generate",
    json={
        "prompt": "Что такое машинное обучение?",
        "max_tokens": 100,
        "temperature": 0.7,
        "top_p": 0.95
    }
)

result = response.json()
print(f"\n📝 Ответ:")
print(result["output"])
print(f"\n📊 Метрики:")
print(f"  - Input tokens: {result['input_tokens']}")
print(f"  - Output tokens: {result['output_tokens']}")
print(f"  - Total tokens: {result['total_tokens']}")
print(f"  - Latency: {result['latency_seconds']:.3f}s")
print(f"  - Tokens/sec: {result['output_tokens'] / result['latency_seconds']:.2f}")

🔍 Проверка здоровья сервиса...
Status: 200
Response: {'status': 'healthy', 'vllm_url': 'https://openrouter.ai/api', 'mlflow_tracking_uri': 'http://mlflow:5000', 'timestamp': 1786427880.9807537}

🤖 Генерация текста...



📝 Ответ:
 Машинное обучение (Machine Learning, ML) - это подраздел искусственного интеллекта, который позволяет компьютерам улучшать свои алгоритмы без явного программирования. Это методология, которая позволяет машинам извлекать знания из данных и использовать их для принятия решений или прогнозирования. Машинное обучение включает в себя множество алгоритмов и техник, которые помогают компьютерам обучаться на основе

📊 Метрики:
  - Input tokens: 8
  - Output tokens: 100
  - Total tokens: 108
  - Latency: 2.639s
  - Tokens/sec: 37.90


### 4.2. Просмотр Prometheus метрик

In [16]:
# Получение Prometheus метрик
response = requests.get(f"{INFERENCE_SERVICE_URL}/metrics")
metrics_text = response.text

# Парсинг основных метрик
print("📊 Prometheus Метрики:\n")

for line in metrics_text.split('\n'):
    if line.startswith('llm_') and not line.startswith('#'):
        print(line)
        
print("\n💡 Эти метрики можно экспортировать в Prometheus и визуализировать в Grafana")

📊 Prometheus Метрики:

llm_requests_total{endpoint="/generate",model="qwen/qwen-2.5-7b-instruct"} 8.0
llm_requests_created{endpoint="/generate",model="qwen/qwen-2.5-7b-instruct"} 1.786420334499122e+09
llm_tokens_total{model="qwen/qwen-2.5-7b-instruct",phase="input"} 33.0
llm_tokens_total{model="qwen/qwen-2.5-7b-instruct",phase="output"} 624.0
llm_tokens_created{model="qwen/qwen-2.5-7b-instruct",phase="input"} 1.7864203348348596e+09
llm_tokens_created{model="qwen/qwen-2.5-7b-instruct",phase="output"} 1.786420335976332e+09
llm_request_latency_seconds_bucket{endpoint="/generate",le="0.05",model="qwen/qwen-2.5-7b-instruct"} 0.0
llm_request_latency_seconds_bucket{endpoint="/generate",le="0.1",model="qwen/qwen-2.5-7b-instruct"} 0.0
llm_request_latency_seconds_bucket{endpoint="/generate",le="0.25",model="qwen/qwen-2.5-7b-instruct"} 0.0
llm_request_latency_seconds_bucket{endpoint="/generate",le="0.5",model="qwen/qwen-2.5-7b-instruct"} 0.0
llm_request_latency_seconds_bucket{endpoint="/generate"

---

## 🧪 Часть 5: Автоматический бенчмаркинг моделей

Скрипт `benchmark_vllm_models.py` позволяет:
- Сравнить несколько моделей на Q&A задачах
- Вычислить F1 Score и Exact Match
- Залогировать результаты в MLflow
- Автоматически выбрать лучшую модель

### Запуск бенчмарка

**Из командной строки:**
```bash
export VLLM_BASE_URL="http://localhost:8000"
export MLFLOW_TRACKING_URI="http://localhost:5000"
export MODELS="facebook/opt-1.3b,EleutherAI/gpt-neo-125M"

python benchmark_vllm_models.py
```

**Или из ноутбука:**

In [17]:
import subprocess
import os
import sys

# Настройка переменных окружения для бенчмарка (бэкенд — OpenRouter)
env = os.environ.copy()
env["VLLM_BASE_URL"] = "https://openrouter.ai/api"
env["VLLM_API_KEY"] = API_KEY               # ключ OpenRouter из .env
env["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
# модели, стабильно работающие через /completions (для сравнения — разного размера)
env["MODELS"] = "qwen/qwen-2.5-7b-instruct,meta-llama/llama-3.2-1b-instruct"
env["PYTHONUTF8"] = "1"                      # чтобы emoji в логах mlflow не ломали stdout на Windows

# Запуск бенчмарка
print("🚀 Запуск бенчмарка...\n")
result = subprocess.run(
    [sys.executable, "benchmark_vllm_models.py"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    env=env
)

print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr[-2000:])


🚀 Запуск бенчмарка...



🏃 View run qwen/qwen-2.5-7b-instruct at: http://localhost:5000/#/experiments/3/runs/0f2e37577e794bf88f2a0194103ea7f0
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run meta-llama/llama-3.2-1b-instruct at: http://localhost:5000/#/experiments/3/runs/64cc043f984e44e181b5ff7745cbcf61
🧪 View experiment at: http://localhost:5000/#/experiments/3

Errors: 7,143 - INFO -   F1 Score: 0.7114 (±0.1239)
2026-08-11 10:58:17,143 - INFO -   Exact Match: 0.0000
2026-08-11 10:58:17,143 - INFO -   Latency (mean): 1.726s
2026-08-11 10:58:17,143 - INFO -   Latency (p95): 2.154s
2026-08-11 10:58:17,143 - INFO -   Success rate: 5/5
2026-08-11 10:58:17,231 - INFO - 
2026-08-11 10:58:17,231 - INFO - Model: meta-llama/llama-3.2-1b-instruct
2026-08-11 10:58:17,231 - INFO - ======================================================================
2026-08-11 10:58:17,374 - INFO - Evaluating model: meta-llama/llama-3.2-1b-instruct
2026-08-11 10:58:17,374 - INFO -   Processing sample 1/5
2026-08-11 

### 5.1. Просмотр результатов бенчмарка

In [18]:
import json
import pandas as pd

# Загрузка сводки результатов
try:
    with open("benchmark_summary.json", "r") as f:
        summary = json.load(f)
    
    print("🏆 Лучшая модель:", summary["best_model"])
    print(f"📈 Лучший F1 Score: {summary['best_score']:.4f}\n")
    
    # Создание таблицы результатов
    results_data = []
    for result in summary["all_results"]:
        model_id = result["model_id"]
        metrics = result["metrics"]
        results_data.append({
            "Model": model_id,
            "F1 Score": f"{metrics['f1_mean']:.4f}",
            "Exact Match": f"{metrics['em_mean']:.4f}",
            "Latency (s)": f"{metrics['latency_mean']:.3f}",
            "Success Rate": f"{metrics['samples_successful']}/{metrics['samples_evaluated']}"
        })
    
    results_df = pd.DataFrame(results_data)
    print("📊 Результаты сравнения моделей:")
    print(results_df.to_string(index=False))
    
    # Чтение имени лучшей модели
    with open("best_model.txt", "r") as f:
        best_model = f.read().strip()
    print(f"\n💾 Лучшая модель сохранена в: best_model.txt")
    print(f"   Содержимое: {best_model}")
    
except FileNotFoundError:
    print("⚠️  Файл benchmark_summary.json не найден.")
    print("   Сначала запустите бенчмарк!")


🏆 Лучшая модель: qwen/qwen-2.5-7b-instruct
📈 Лучший F1 Score: 0.7114

📊 Результаты сравнения моделей:
                           Model F1 Score Exact Match Latency (s) Success Rate
       qwen/qwen-2.5-7b-instruct   0.7114      0.0000       1.726          5/5
meta-llama/llama-3.2-1b-instruct   0.4706      0.2000       1.490          5/5

💾 Лучшая модель сохранена в: best_model.txt
   Содержимое: qwen/qwen-2.5-7b-instruct


---

## 📝 Выводы и заключение

### Что было реализовано:

#### 1. **vLLM Deployment** ✅
- Локальный LLM сервер с OpenAI-совместимым API
- Поддержка различных моделей
- CPU и GPU режимы работы

#### 2. **API Integration** ✅
- HTTP запросы через `requests`
- OpenAI client библиотека
- FastAPI production сервис с метриками

#### 3. **MLflow Integration** ✅
- Кастомные метрики для оценки качества
- LLM-as-a-Judge паттерн
- Автоматическое логирование экспериментов
- Артефакты и параметры

#### 4. **Production Features** ✅
- Prometheus метрики (requests, tokens, latency, errors)
- Health checks
- Error handling
- OpenAPI документация

#### 5. **Автоматический бенчмаркинг** ✅
- Сравнение моделей на Q&A задачах
- F1 Score и Exact Match метрики
- Автоматический выбор лучшей модели
- Интеграция с MLflow

### Ключевые метрики:

- **F1 Score**: token-level совпадение с эталонным ответом
- **Exact Match**: точное совпадение ответа
- **Latency**: время генерации ответа
- **Tokens/Second**: скорость генерации
- **Success Rate**: процент успешных запросов

### Production-Ready компоненты:

1. **inference_service.py** - FastAPI сервис с метриками
2. **benchmark_vllm_models.py** - автоматизированное тестирование
3. **start_vllm_server.py** - удобный запуск vLLM
4. **test_vllm_server.py** - проверка работоспособности

### Дальнейшее развитие:

- [ ] Kubernetes deployment (Helm charts)
- [ ] Distributed tracing (Jaeger/Zipkin)
- [ ] Advanced caching strategies
- [ ] A/B testing framework
- [ ] Continuous monitoring & alerting
- [ ] Load testing & benchmarking on larger datasets

### Полезные ссылки:

- [vLLM Documentation](https://docs.vllm.ai/)
- [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html)
- [FastAPI](https://fastapi.tiangolo.com/)
- [Prometheus Metrics](https://prometheus.io/docs/concepts/metric_types/)
- [Reference Implementation](https://github.com/Ilia2704/llm_mlflow)